# 62 — Grand Ensemble v7

Incorporates all new models from nb37–nb61 plus existing models from nb01–nb36.

Improvements over v6:
- External data models: nb55 (all external), nb57 (PubChem), nb58 (BindingDB)
- 10-head Chemprop: nb56
- Cliff-focused: nb40, nb45, nb59, nb60
- Collapse criterion: skip models where test_std < 0.4 * train_std
- Cliff-focused sub-ensemble: v7c uses only models with cliff_member RAE < 0.6
- Target: OOF RAE < 0.50 (vs v6 0.5281)

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
V6_RAE = 0.5281  # benchmark to beat

tr = load_train()
te = load_test()
y_tr = tr['pec50'].values.astype(np.float32)

scaffolds = tr['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

train_std = float(y_tr.std())
print(f'Train: {len(tr):,}  Test: {len(te):,}  Train pEC50 std: {train_std:.4f}')
print('Setup complete.')

Train: 4,139  Test: 513  Train pEC50 std: 1.1215
Setup complete.


## 1. Load all OOF + test arrays

In [2]:
# ── Explicit model registry (all nb01–nb61 models) ────────────────────────────
# (oof_stem, te_stem, label)
MODEL_REGISTRY = [
    # nb01–nb36 models (Grand v6 base)
    ('oof_lgbm_base',              'te_lgbm_base',              'lgbm_base'),
    ('oof_lgbm_tuned',             'te_lgbm_tuned',             'lgbm_tuned'),
    ('oof_knn',                    'te_knn',                    'knn'),
    ('oof_chemberta',              'te_chemberta',              'chemberta_mlm'),
    ('oof_chemberta_mtr',          'te_chemberta_mtr',          'chemberta_mtr'),
    ('oof_bert_smiles',            'te_bert_smiles',            'bert_smiles'),
    ('oof_unimol',                 'te_unimol',                 'unimol'),
    ('oof_grover',                 'te_grover',                 'grover_base'),
    ('oof_grover_large',           'te_grover_large',           'grover_large'),
    ('oof_singleconc',             'te_singleconc',             'singleconc_lgbm'),
    ('oof_nr_weighted',            'te_nr_weighted',            'nr_weighted_lgbm'),
    ('oof_morgan_esm2_nr',         'te_morgan_esm2_nr',         'morgan_esm2_nr'),
    ('oof_chemberta_esm2_nr',      'te_chemberta_esm2_nr',      'chemberta_esm2_nr'),
    ('oof_morgan_protbert_nr',     'te_morgan_protbert_nr',     'morgan_protbert_nr'),
    ('oof_crossattn_grover_esm2',  'te_crossattn_grover_esm2',  'crossattn_grover_esm2'),
    ('oof_chemprop_aux',           'te_chemprop_aux',           'chemprop_aux'),
    ('oof_chemprop',               'te_chemprop',               'chemprop'),
    # nb37–nb61 new models
    ('oof_cliff_weighted',         'te_cliff_weighted',         'cliff_weighted_lgbm'),
    ('oof_pairwise_ranking',       'te_pairwise_ranking',       'pairwise_ranking_lgbm'),
    ('oof_lgbm_all_external',      'te_lgbm_all_external',      'lgbm_all_external'),
    ('oof_chemprop_10head',        'te_chemprop_10head',        'chemprop_10head'),
    ('oof_lgbm_pubchem',           'te_lgbm_pubchem',           'lgbm_pubchem'),
    ('oof_lgbm_bindingdb',         'te_lgbm_bindingdb',         'lgbm_bindingdb'),
    ('oof_lgbm_cliff_oversample',  'te_lgbm_cliff_oversample',  'lgbm_cliff_oversample'),
    ('oof_deep_graph_cliff',       'te_deep_graph_cliff',       'deep_graph_cliff'),
]

# ── Exclusion rules ────────────────────────────────────────────────────────────
EXCLUDE_STEMS = {
    'oof_aux_features',      # train-only features (emax/pec50_se) — known to collapse test
    'oof_grand_v6',          # previous ensemble OOF
    'oof_grand_v7',          # this ensemble OOF (prevent recursion)
}

loaded_all = []   # all passing models
loaded_new = []   # nb37–nb61 only

NEW_MODEL_STEMS = {m[0] for m in MODEL_REGISTRY[17:]}  # nb37+ models

print(f'\nLoading models...')
print(f'{"Model":45s} {"OOF RAE":>9} {"Test std":>9} {"Status"}')
print('-' * 80)

for oof_stem, te_stem, label in MODEL_REGISTRY:
    if oof_stem in EXCLUDE_STEMS:
        print(f'{label:45s} {"":>9} {"":>9} EXCLUDED (explicit)')
        continue

    op = DATA_PROCESSED / f'{oof_stem}.npy'
    tp = DATA_PROCESSED / f'{te_stem}.npy'

    if not op.exists():
        print(f'{label:45s} {"":>9} {"":>9} MISSING ({oof_stem}.npy)')
        continue

    try:
        oof_arr = np.load(str(op))
    except Exception as e:
        print(f'{label:45s} {"":>9} {"":>9} ERROR ({e})')
        continue

    if len(oof_arr) != len(tr):
        print(f'{label:45s} {"":>9} {"":>9} WRONG SIZE ({len(oof_arr)}!={len(tr)})')
        continue

    # Check test predictions
    if not tp.exists():
        print(f'{label:45s} {"":>9} {"":>9} MISSING ({te_stem}.npy)')
        continue
    try:
        te_arr = np.load(str(tp))
    except Exception:
        print(f'{label:45s} {"":>9} {"":>9} TEST FILE ERROR')
        continue

    # Collapse criterion: test_std < 0.4 * train_std
    te_std = float(te_arr.std())
    if te_std < 0.4 * train_std:
        print(f'{label:45s} {"":>9} {te_std:>9.4f} COLLAPSED (test_std too low)')
        continue

    oof_rae = rae_fn(y_tr, oof_arr)
    entry = {'oof': oof_arr, 'te': te_arr, 'label': label, 'oof_rae': oof_rae}

    loaded_all.append(entry)
    is_new = oof_stem in NEW_MODEL_STEMS
    if is_new:
        loaded_new.append(entry)

    print(f'{label:45s} {oof_rae:>9.4f} {te_std:>9.4f} OK{" (NEW)" if is_new else ""}')

print(f'\nTotal loaded: {len(loaded_all)} (new: {len(loaded_new)})')

# Also scan for any oof_*.npy not in registry
found_extra = []
for p in sorted(DATA_PROCESSED.glob('oof_*.npy')):
    stem = p.stem
    if stem in EXCLUDE_STEMS:
        continue
    if not any(stem == m[0] for m in MODEL_REGISTRY):
        te_p = DATA_PROCESSED / p.name.replace('oof_', 'te_')
        if te_p.exists():
            try:
                oof_arr = np.load(str(p))
                te_arr  = np.load(str(te_p))
                if len(oof_arr) == len(tr) and te_arr.std() >= 0.4 * train_std:
                    label = stem[4:]  # strip 'oof_'
                    entry = {'oof': oof_arr, 'te': te_arr, 'label': label,
                              'oof_rae': rae_fn(y_tr, oof_arr)}
                    found_extra.append(entry)
                    loaded_all.append(entry)
                    print(f'  EXTRA: {label:40s} OOF RAE={entry["oof_rae"]:.4f}')
            except Exception:
                pass

if found_extra:
    print(f'\n{len(found_extra)} extra models found outside registry.')

if len(loaded_all) < 2:
    raise RuntimeError(f'Need at least 2 models for ensemble. Only {len(loaded_all)} loaded. '
                       'Run earlier notebooks first.')


Loading models...
Model                                           OOF RAE  Test std Status
--------------------------------------------------------------------------------
lgbm_base                                        0.5600    0.6494 OK
lgbm_tuned                                       0.5394    0.6712 OK
knn                                                        0.3414 COLLAPSED (test_std too low)
chemberta_mlm                                    0.6782    0.4721 OK
chemberta_mtr                                    0.5993    0.5561 OK
bert_smiles                                      0.7150    0.4499 OK
unimol                                           0.7008    0.4531 OK
grover_base                                      0.6355    0.5532 OK
grover_large                                     0.6295    0.5790 OK
singleconc_lgbm                                  0.6003    0.5023 OK
nr_weighted_lgbm                                 0.5964    0.6230 OK
morgan_esm2_nr                            

## 2. Load cliff breakdown for sub-ensemble selection

In [3]:
cliff_breakdown_path = DATA_PROCESSED / 'cliff_model_breakdown.parquet'
good_cliff_models = set()

if cliff_breakdown_path.exists():
    cliff_bkdn = pd.read_parquet(cliff_breakdown_path)
    # Models with cliff_member RAE < 0.6
    if 'cliff_rae' in cliff_bkdn.columns and 'model' in cliff_bkdn.columns:
        good_cliff_models = set(
            cliff_bkdn.loc[cliff_bkdn['cliff_rae'].notna() &
                            (cliff_bkdn['cliff_rae'] < 0.6), 'model']
        )
        print(f'Models with cliff_member RAE < 0.6 (from nb61): {len(good_cliff_models)}')
        print(f'  {sorted(good_cliff_models)}')
    else:
        print('cliff_model_breakdown.parquet lacks expected columns')
else:
    print('cliff_model_breakdown.parquet not found (run nb61 first).')
    print('Sub-ensemble v7c will be skipped or will use all models.')

cliff_model_breakdown.parquet not found (run nb61 first).
Sub-ensemble v7c will be skipped or will use all models.


## 3. Nested scaffold CV meta-learner

In [4]:
ALPHAS = np.logspace(-4, 1, 30)
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]


def nested_ensemble_cv(loaded_models, splits, y_tr, label=''):
    """Nested scaffold CV meta-learner (ElasticNetCV on OOF stacks)."""
    if len(loaded_models) < 2:
        print(f'  {label}: need >= 2 models, only {len(loaded_models)} — skipping')
        return None, None

    oof_matrix = np.column_stack([m['oof'] for m in loaded_models])   # (N, n_models)
    oof_meta   = np.full(len(y_tr), np.nan)

    for fold_i, (tr_idx, va_idx) in enumerate(splits):
        X_inner = oof_matrix[tr_idx]
        y_inner = y_tr[tr_idx]
        X_val   = oof_matrix[va_idx]

        scaler = StandardScaler()
        X_inner_s = scaler.fit_transform(X_inner)
        X_val_s   = scaler.transform(X_val)

        en = ElasticNetCV(alphas=ALPHAS, l1_ratio=L1_RATIOS, cv=5,
                           fit_intercept=True, max_iter=5000, random_state=SEED)
        en.fit(X_inner_s, y_inner)
        oof_meta[va_idx] = en.predict(X_val_s)

    nested_rae = rae_fn(y_tr, oof_meta)
    return oof_meta, nested_rae


print('Running nested CV for all variants...')

# Variant v7a: new models only
if len(loaded_new) >= 2:
    print(f'\nVariant v7a: {len(loaded_new)} new models only (nb37–nb61)')
    oof_v7a, rae_v7a = nested_ensemble_cv(loaded_new, splits, y_tr, 'v7a')
    print(f'  v7a nested CV RAE: {rae_v7a:.4f}')
else:
    print(f'v7a: not enough new models ({len(loaded_new)} < 2) — skipping')
    oof_v7a, rae_v7a = None, float('inf')

# Variant v7b: all models (new + old)
print(f'\nVariant v7b: all {len(loaded_all)} models (nb01–nb61)')
oof_v7b, rae_v7b = nested_ensemble_cv(loaded_all, splits, y_tr, 'v7b')
print(f'  v7b nested CV RAE: {rae_v7b:.4f}')

# Variant v7c: cliff-focused subset
cliff_focused = [m for m in loaded_all if m['label'] in good_cliff_models] if good_cliff_models else []
if len(cliff_focused) >= 2:
    print(f'\nVariant v7c: {len(cliff_focused)} cliff-focused models (cliff_rae < 0.6)')
    oof_v7c, rae_v7c = nested_ensemble_cv(cliff_focused, splits, y_tr, 'v7c')
    print(f'  v7c nested CV RAE: {rae_v7c:.4f}')
else:
    print(f'\nVariant v7c: not enough cliff-focused models ({len(cliff_focused)}) — using v7b')
    oof_v7c, rae_v7c = oof_v7b, rae_v7b

print(f'\n=== Variant Comparison ===')
print(f'  v7a (new only):        {rae_v7a:.4f}' if rae_v7a < float('inf') else '  v7a: skipped')
print(f'  v7b (all):             {rae_v7b:.4f}')
print(f'  v7c (cliff-focused):   {rae_v7c:.4f}')
print(f'  v6 benchmark:          {V6_RAE:.4f}')

Running nested CV for all variants...

Variant v7a: 4 new models only (nb37–nb61)


  v7a nested CV RAE: 0.5605

Variant v7b: all 32 models (nb01–nb61)


  v7b nested CV RAE: 0.5189

Variant v7c: not enough cliff-focused models (0) — using v7b

=== Variant Comparison ===
  v7a (new only):        0.5605
  v7b (all):             0.5189
  v7c (cliff-focused):   0.5189
  v6 benchmark:          0.5281


## 4. Weight analysis

In [5]:
# ── Full-data ElasticNetCV for final weights (use best variant's model set) ───
best_rae = min(r for r in [rae_v7a, rae_v7b, rae_v7c] if r < float('inf'))

if rae_v7a == best_rae and oof_v7a is not None:
    best_variant = 'v7a'
    best_models = loaded_new
    best_oof = oof_v7a
elif rae_v7c == best_rae and oof_v7c is not None and rae_v7c < rae_v7b:
    best_variant = 'v7c'
    best_models = cliff_focused if len(cliff_focused) >= 2 else loaded_all
    best_oof = oof_v7c
else:
    best_variant = 'v7b'
    best_models = loaded_all
    best_oof = oof_v7b

print(f'Best variant: {best_variant}  (nested CV RAE={best_rae:.4f})')

# Fit full-data ElasticNetCV to get weights for test prediction
oof_matrix_best = np.column_stack([m['oof'] for m in best_models])
te_matrix_best  = np.column_stack([m['te'] for m in best_models])
labels_best     = [m['label'] for m in best_models]

scaler_full = StandardScaler()
X_full_s    = scaler_full.fit_transform(oof_matrix_best)
X_te_s      = scaler_full.transform(te_matrix_best)

en_full = ElasticNetCV(alphas=ALPHAS, l1_ratio=L1_RATIOS, cv=5,
                        fit_intercept=True, max_iter=5000, random_state=SEED)
en_full.fit(X_full_s, y_tr)

# Weight analysis
coefs = en_full.coef_
total_abs = np.abs(coefs).sum()
weight_pct = np.abs(coefs) / (total_abs + 1e-12) * 100
n_nonzero = (coefs != 0).sum()

print(f'\nFull-data ElasticNetCV: alpha={en_full.alpha_:.5f}  l1={en_full.l1_ratio_:.2f}')
print(f'Non-zero coefficients: {n_nonzero}/{len(labels_best)}')

print('\nFinal model weights:')
order = np.argsort(weight_pct)[::-1]
for i in order:
    if weight_pct[i] > 0.01:
        sign = '+' if coefs[i] > 0 else '-'
        is_new = labels_best[i] in {m['label'] for m in loaded_new}
        tag = '(NEW)' if is_new else ''
        print(f'  {labels_best[i]:45s} {sign}{weight_pct[i]:5.1f}%  {tag}')

# Fraction of weight from cliff-specific models
cliff_model_labels = {'cliff_weighted_lgbm', 'pairwise_ranking_lgbm',
                       'lgbm_cliff_oversample', 'deep_graph_cliff', 'chemprop_10head'}
cliff_weight_pct = sum(
    weight_pct[i] for i, l in enumerate(labels_best) if l in cliff_model_labels
)
new_weight_pct = sum(
    weight_pct[i] for i, l in enumerate(labels_best)
    if l in {m['label'] for m in loaded_new}
)
print(f'\nWeight from cliff-specific models: {cliff_weight_pct:.1f}%')
print(f'Weight from new models (nb37+):    {new_weight_pct:.1f}%')

Best variant: v7b  (nested CV RAE=0.5189)



Full-data ElasticNetCV: alpha=0.01172  l1=1.00
Non-zero coefficients: 6/32

Final model weights:
  lgbm_tuned                                    + 45.4%  
  deep_ensemble                                 + 28.7%  
  chemprop_aux                                  + 21.3%  
  singleconc_lgbm                               +  2.5%  
  unimol                                        +  2.0%  
  chemberta_mlm                                 +  0.1%  

Weight from cliff-specific models: 0.0%
Weight from new models (nb37+):    0.0%


## 5. Test predictions

In [6]:
# ── Generate test predictions from best variant ─────────────────────────────
te_preds = en_full.predict(X_te_s)
te_preds = np.clip(te_preds, float(y_tr.min()) - 0.5, float(y_tr.max()) + 0.5)

# Compare test distribution to v6b
v6b_te_path = DATA_PROCESSED / 'te_grand_v6.npy'
if v6b_te_path.exists():
    te_v6b = np.load(str(v6b_te_path))
    print(f'Test distribution comparison:')
    print(f'  v6 test std:   {te_v6b.std():.4f}  mean={te_v6b.mean():.3f}')
    print(f'  v7 test std:   {te_preds.std():.4f}  mean={te_preds.mean():.3f}')
    if te_preds.std() > te_v6b.std():
        print('  v7 has BROADER test prediction range (good — better calibration)')
    else:
        print('  v7 has narrower test prediction range')
else:
    print(f'Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.4f}  '
          f'min={te_preds.min():.3f}  max={te_preds.max():.3f}')

# Save OOF
np.save(DATA_PROCESSED / 'oof_grand_v7.npy', best_oof)
print(f'\nSaved oof_grand_v7.npy  (OOF RAE = {rae_fn(y_tr, best_oof):.4f})')

Test distribution comparison:
  v6 test std:   0.1176  mean=4.619
  v7 test std:   0.6255  mean=4.762
  v7 has BROADER test prediction range (good — better calibration)

Saved oof_grand_v7.npy  (OOF RAE = 0.5189)


## 6. Save submission

In [7]:
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50':         te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'

out_path = SUBMISSIONS / '62_grand_v7.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print(f'\n{"="*60}')
print('Grand Ensemble v7 Summary')
print(f'{"="*60}')
print(f'  Total models evaluated:     {len(loaded_all)}')
print(f'  New models (nb37–nb61):     {len(loaded_new)}')
print(f'  Cliff-focused models:       {len(cliff_focused)}')
print(f'  Best variant:               {best_variant}')
print(f'  Nested CV RAE (best):       {best_rae:.4f}')
print(f'  v6b benchmark:              {V6_RAE:.4f}')
delta = best_rae - V6_RAE
print(f'  Delta vs v6b:               {delta:+.4f} ({"IMPROVEMENT" if delta < 0 else "REGRESSION"})')
print(f'  Target (< 0.50):            {"MET" if best_rae < 0.50 else "NOT YET MET"}')
print(f'  Test pred std:              {te_preds.std():.4f}')
print(f'{"="*60}')

print('\nVariant comparison:')
print(f'  v7a (new only):            {rae_v7a:.4f}' if rae_v7a < float('inf') else '  v7a: not enough models')
print(f'  v7b (all):                 {rae_v7b:.4f}')
print(f'  v7c (cliff-focused):       {rae_v7c:.4f}')

sub['pEC50'].describe().round(3)

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\62_grand_v7.csv

Grand Ensemble v7 Summary
  Total models evaluated:     32
  New models (nb37–nb61):     4
  Cliff-focused models:       0
  Best variant:               v7b
  Nested CV RAE (best):       0.5189
  v6b benchmark:              0.5281
  Delta vs v6b:               -0.0092 (IMPROVEMENT)
  Target (< 0.50):            NOT YET MET
  Test pred std:              0.6255

Variant comparison:
  v7a (new only):            0.5605
  v7b (all):                 0.5189
  v7c (cliff-focused):       0.5189


count    513.000
mean       4.762
std        0.626
min        2.161
25%        4.390
50%        4.909
75%        5.239
max        5.849
Name: pEC50, dtype: float64